**Note:** This notebook is designed for **Google Colab**.

If you see the Colab logo <span style='vertical-align:bottom;'><img src='https://colab.research.google.com/img/colab_favicon_256px.png' width='40' alt='Colab logo'></span> in the top-left corner, you're all set! Please **continue**.

If you don't see the logo (e.g., you are on GitHub), please click the button below to open it in the correct environment:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mparrott-at-wiris/aimodelshare/blob/master/notebooks/qmss_climate_competition_notebook.ipynb)

# **QMSS Climate Change ML Competition: Build & Submit Custom Models**

Welcome to the **QMSS Climate Change ML Competition**.

**Why this matters:**
Buildings account for nearly 40% of global energy consumption and contribute roughly 30% of global greenhouse gas emissions. Identifying which buildings are high energy users enables targeted retrofits and meaningful reductions in carbon emissions. Your models can help pinpoint the buildings where intervention will have the greatest climate impact.

**The Goal:**
Train a model to predict which buildings are high energy users (`high_energy_usage = 1`) using building characteristics, location, and climate features.

**Who is this for?**
This notebook is designed for participants with Python experience (e.g., Scikit-Learn, TensorFlow, PyTorch). You will build, train, and submit your own machine learning models directly to the competition leaderboard.

## Competition Awards & Rules

### Individual & Team Tracks
This competition has **two separate award tracks**:
- **Individual Track** — Your modelshare.ai username is automatically captured when you log in. Every submission is linked to your individual account.
- **Team Track** — If you are working as a team, set the `"Team"` field in your submission to your team name. All team members should use the **same team name** when submitting.

You can participate in both tracks simultaneously. Digital awards are given separately for individuals and teams.

### Digital Awards
All awards are **digital awards**. Winners will be publicized by the QMSS Program.
- **1st, 2nd, 3rd Place** — Top three submissions on the final leaderboard receive digital podium awards (awarded separately for individuals and teams).
- **Gold Medal** — Top 1% of all submissions
- **Silver Medal** — Top 5% of all submissions
- **Bronze Medal** — Top 10% of all submissions

Medals are awarded in addition to podium prizes (e.g., a 1st-place finisher also earns a Gold Medal).

### Scoring
Your submissions are scored on the held-out test set using the following classification metrics:
- **F1 Score** — harmonic mean of precision and recall
- **Accuracy** — overall proportion of correct predictions
- **Precision** — proportion of predicted positives that are truly positive
- **Recall** — proportion of actual positives correctly identified

The leaderboard displays all four metrics so you can evaluate your model's performance in detail. **Final award winners will be ranked by F1 Score.**

### Tie-Breaker
If two submissions achieve the same F1 score, the submission with the **earliest timestamp** wins.

### Submission Rules
- Each submission must be a 1D array of **25,000 binary predictions** (0 or 1), one per test sample.
- You may submit **multiple times** — the leaderboard tracks all submissions.
- **Submitting as an individual?** Leave the `"Team"` field empty (`""`) or remove it entirely. Your username is captured automatically.
- **Submitting as a team?** Set the `"Team"` field to your team name. All team members should use the same name.

## Quick Start Guide

To participate in the competition, complete these 5 steps:

1.  **Install Libraries:** Run the setup cell to install `aimodelshare`.
2.  **Get the Data:** Run the data loading cell to retrieve the pre-split training and testing data.
3.  **Train Your Model:** Use the provided Scikit-Learn Pipeline example or write your own custom training code.
4.  **Connect:** Link this notebook to the QMSS Climate Competition Leaderboard.
5.  **Submit:** Send your predictions to the leaderboard to see your score.

**Ready? Click the Play Button on the first code cell below to get started.**

---
# **Step 1: Installation**

We need to install the `aimodelshare` library to connect to the competition backend.

In [ ]:
# Install the aimodelshare library
print("Installing required libraries...")
!pip install aimodelshare --upgrade -q --no-warn-script-location > /dev/null 2>&1
print("Installation complete!")

---
# **Step 2: Load Data**

We will load the pre-split competition data directly from the competition URLs.

* **train.csv** — 75,000 building records with features + target label (`high_energy_usage`)
* **X_test.csv** — 25,000 building records with features only (no target label)

You will train your model on the training data and generate predictions on X_test. The true test labels are held by the leaderboard — your score is computed when you submit.

### Feature Guide

The dataset contains **45 features** across three categories:

| Category | Features | Description |
|---|---|---|
| **Building characteristics** | `facility_type`, `floor_area`, `year_built`, `building_class` | Building use type (e.g., office, retail), gross floor area (sq ft), construction year, and building classification |
| **Location** | `State_Factor`, `ELEVATION` | US state identifier and elevation in meters above sea level |
| **Climate — monthly temps** | `january_min_temp` ... `december_max_temp` (36 features) | Min, average, and max temperature for each month (°F) |
| **Climate — annual** | `avg_temp`, `heating_degree_days`, `cooling_degree_days` | Annual average temperature and degree-day measures of heating/cooling energy demand |

**Target variable:** `high_energy_usage` — binary label where 1 = high energy user, 0 = not a high energy user.

For detailed feature descriptions including value distributions and ranges, see the full [Dataset Description](https://github.com/mparrott-at-wiris/aimodelshare/blob/master/datasets/qmss_climate/QMSS_COMPETITION_DATA_DESCRIPTION.md).

In [ ]:
import pandas as pd

# 1. Load training data (features + target)
DATA_BASE = "https://raw.githubusercontent.com/mparrott-at-wiris/aimodelshare/master/datasets/qmss_climate"
train_df = pd.read_csv(f"{DATA_BASE}/train.csv")

# 2. Load test data (features only — no target labels)
X_test = pd.read_csv(f"{DATA_BASE}/X_test.csv")

# 3. Separate training features and target
NUMERIC_COLS = [
    "floor_area", "year_built", "ELEVATION",
    "heating_degree_days", "cooling_degree_days", "avg_temp",
    "january_min_temp", "january_avg_temp", "january_max_temp",
    "february_min_temp", "february_avg_temp", "february_max_temp",
    "march_min_temp", "march_avg_temp", "march_max_temp",
    "april_min_temp", "april_avg_temp", "april_max_temp",
    "may_min_temp", "may_avg_temp", "may_max_temp",
    "june_min_temp", "june_avg_temp", "june_max_temp",
    "july_min_temp", "july_avg_temp", "july_max_temp",
    "august_min_temp", "august_avg_temp", "august_max_temp",
    "september_min_temp", "september_avg_temp", "september_max_temp",
    "october_min_temp", "october_avg_temp", "october_max_temp",
    "november_min_temp", "november_avg_temp", "november_max_temp",
    "december_min_temp", "december_avg_temp", "december_max_temp",
]
CATEGORICAL_COLS = ["facility_type", "building_class", "State_Factor"]
ALL_FEATURES = NUMERIC_COLS + CATEGORICAL_COLS

X_train = train_df[ALL_FEATURES]
y_train = train_df["high_energy_usage"]

print(f"Training data: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test data: {X_test.shape[0]} samples (no labels — scored by leaderboard)")
print(f"\nTraining target distribution:\n{y_train.value_counts().to_dict()}")
print("\nFirst 5 rows of training data:")
X_train.head()

---
# **Step 3: Train Your Model**

Below is a **baseline example** using a Scikit-Learn Pipeline with Logistic Regression. This pipeline:
1.  **Scales** numerical columns (floor area, temperatures, degree days, etc.).
2.  **One-Hot Encodes** categorical columns (facility type, building class, state).
3.  **Trains** a Logistic Regression classifier.

**To improve your score**, consider:
- Replacing the classifier with a more powerful model (RandomForest, GradientBoosting, XGBoost, Neural Network, etc.)
- Engineering new features from existing ones (e.g., temperature ranges, climate-building interactions)
- Tuning hyperparameters with cross-validation
- Handling class imbalance if present in the training data

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. Define Transformers
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 2. Create Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERIC_COLS),
        ('cat', categorical_transformer, CATEGORICAL_COLS)
    ])

# 3. Create Pipeline (Preprocessor + Model)
# Replace LogisticRegression with any model you want to try
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# 4. Train the pipeline on training data
pipeline.fit(X_train, y_train)

# 5. Generate predictions on the test set
predictions = pipeline.predict(X_test)

print(f"Model trained! Predictions generated for {len(predictions)} test samples.")
print(f"Training accuracy: {pipeline.score(X_train, y_train):.4f}")
print("Submit your predictions in the next step to see your test score on the leaderboard.")

---
# **Step 4: Connect to the Leaderboard**

This step connects your notebook to the QMSS Climate Change ML Competition backend.

*Note: You will be prompted to enter a username and password. If you don't have one, you will need to create one at [modelshare.ai](https://www.modelshare.ai)*

In [ ]:
from aimodelshare.aws import set_credentials
from aimodelshare.playground import Competition
import os

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# The specific Model Playground URL for the QMSS Climate Competition
my_playground_url = "https://br3t6wp6yg.execute-api.us-east-1.amazonaws.com/prod/m"

# Set your credentials (pop-up will appear)
set_credentials(apiurl=my_playground_url)

# Generate your session access token
token = os.getenv("AWS_TOKEN")

# Connect to the competition
playground = Competition(my_playground_url)

---
# **Step 5: Submit & Check Results**

Submit your predictions to the leaderboard.

- **Individual submission:** Leave the `"Team"` field as `""` (your username is captured automatically).
- **Team submission:** Set `"Team"` to your team name.

In [ ]:
# 1. Submit your predictions
playground.submit_model(
    model=None,
    preprocessor=None,
    prediction_submission=predictions,
    token=token,
    input_dict={
        "Team": "",  # Leave empty for individual submission, or set your team name (e.g., "Climate Crusaders")
        "description": "Logistic Regression with Sklearn Pipeline",
        "tags": "sklearn, logistic_regression, climate, baseline"
    }
)

print("Predictions submitted successfully!")

In [ ]:
# 2. Check the leaderboard
print("Loading leaderboard...")
leaderboard = playground.get_leaderboard()
playground.stylize_leaderboard(leaderboard)